In [1]:
#!/usr/bin/env python3
"""
Twitch Gift Sub Uplift Simulation Model
Reverse-Engineered Net Platform Revenue Optimization Script
"""

from dataclasses import dataclass
from typing import List


# ==============================================================================
# 1. CHANNEL & PLATFORM CONFIGURATION (EDIT GLOBAL CONSTANTS HERE)
# ==============================================================================
@dataclass
class ChannelConfig:
    sub_price: float = 4.99               # Base Tier-1 sub price in USD
    streamer_sub_split: float = 0.50      # 0.50 = Standard (50/50), 0.70 = Plus Program (70/30)
    cpm: float = 12.00                    # Channel gross ad CPM in USD
    streamer_ad_split: float = 0.55       # Creator ad share (Twitch retains 1 - split = 45%)
    ads_per_hour: float = 6.0             # 6 x 30-sec spots = 3 minutes of ads/hour

    @property
    def twitch_net_sub_take(self) -> float:
        """Twitch's retained gross margin per paid sub month."""
        return round(self.sub_price * (1.0 - self.streamer_sub_split), 2)

    @property
    def twitch_net_ad_take_per_imp(self) -> float:
        """Twitch's retained ad revenue per single impression."""
        return (self.cpm / 1000.0) * (1.0 - self.streamer_ad_split)


# ==============================================================================
# 2. CANDIDATE DEFINITION & SAMPLE USERS (EDIT CANDIDATES HERE)
# ==============================================================================
@dataclass
class Candidate:
    name: str                       # Candidate display title
    status: str                     # Qualitative state in stream/chat
    watch_hours: float              # Expected monthly hours watched on channel
    p_organic: float                # P(organic | no gift) - organic purchase probability
    p_resub_given_gift: float       # P(resub | gift) - probability user pays after 30-day gift
    ltv_months: float               # Expected retention lifespan (months) if they convert
    uses_adblock: bool              # True (alpha = 1, loses $0 ad rev) / False (alpha = 0)
    gifter_reaction_value: float    # Dollar utility of immediate chat validation for the gifter
    retention_ad_lift: float = 0.0  # Optional: ad revenue lift on other channels via retention


# Pre-populated placeholder candidates from the empirical analysis
CANDIDATES: List[Candidate] = [
    Candidate(
        name="Candidate A: The Cold/Disengaged Account",
        status="Offline follower, not in chat.",
        watch_hours=0.5,
        p_organic=0.0005,           # 0.05%
        p_resub_given_gift=0.0030,  # 0.30%
        ltv_months=3.0,
        uses_adblock=False,
        gifter_reaction_value=0.00,
        retention_ad_lift=0.00,
    ),
    Candidate(
        name="Candidate B: The High-Engagement Ad-Tolerant Chatter (No Ad-Block)",
        status="In chat, active chatter, watches 35 hrs/mo on mobile (no ad-block).",
        watch_hours=35.0,
        p_organic=0.1200,           # 12.0%
        p_resub_given_gift=0.1800,  # 18.0%
        ltv_months=3.2,
        uses_adblock=False,
        gifter_reaction_value=0.75,
        retention_ad_lift=0.00,
    ),
    Candidate(
        name="Candidate C: The Persuadable Lurker/Chatter with Ad-Block",
        status="Live in stream right now, watches 15 hrs/mo, desktop web with uBlock.",
        watch_hours=15.0,
        p_organic=0.0100,           # 1.0%
        p_resub_given_gift=0.0850,  # 8.5%
        ltv_months=3.0,
        uses_adblock=True,
        gifter_reaction_value=0.40,
        retention_ad_lift=0.00,
    ),
]


# ==============================================================================
# 3. ALGORITHMIC SCORING ENGINE
# ==============================================================================
def evaluate_candidate(c: Candidate, config: ChannelConfig):
    # 1. Incremental subscription uplift probability
    uplift_prob = c.p_resub_given_gift - c.p_organic

    # 2. Expected net subscription margin gain
    sub_uplift_margin = uplift_prob * (c.ltv_months * config.twitch_net_sub_take)

    # 3. Forfeited ad margin (0 if adblocker is active)
    ad_multiplier = 0.0 if c.uses_adblock else 1.0
    lost_ad_margin = (
        ad_multiplier
        * c.watch_hours
        * config.ads_per_hour
        * config.twitch_net_ad_take_per_imp
    )

    # 4. Total Expected Net Platform EV
    net_ev = (
        sub_uplift_margin
        - lost_ad_margin
        + c.retention_ad_lift
        + c.gifter_reaction_value
    )

    return {
        "candidate": c,
        "uplift_prob": uplift_prob,
        "sub_uplift_margin": sub_uplift_margin,
        "lost_ad_margin": lost_ad_margin,
        "net_ev": net_ev,
    }


def run_simulation(candidates: List[Candidate], config: ChannelConfig):
    evaluations = [evaluate_candidate(c, config) for c in candidates]
    
    # Sort descending by Net EV
    ranked = sorted(evaluations, key=lambda x: x["net_ev"], reverse=True)
    winner_name = ranked[0]["candidate"].name

    # Print candidate breakdowns
    for item in evaluations:
        c: Candidate = item["candidate"]
        u_pct = item["uplift_prob"] * 100.0
        p_org_pct = c.p_organic * 100.0
        p_resub_pct = c.p_resub_given_gift * 100.0

        print(c.name)
        print(f"- Status: {c.status}")
        print(f"- Watch Time: {c.watch_hours:.1f} hrs/mo.")
        print(
            f"- P(organic) = {p_org_pct:.2f}% | "
            f"P(resub|gift) = {p_resub_pct:.2f}% | "
            f"Uplift = +{u_pct:.2f}%"
        )

        if c.uses_adblock:
            print("- Ad-block: Yes (α = 1) | Lost Ad Margin: $0.00")
        else:
            print(
                f"- Ad-block: No (α = 0) | Lost Ad Margin: "
                f"{c.watch_hours:.1f} * {config.ads_per_hour:.0f} * "
                f"${config.twitch_net_ad_take_per_imp:.4f} = -${item['lost_ad_margin']:.2f}"
            )

        print(
            f"- Sub Uplift: {item['uplift_prob']:.4f} * "
            f"({c.ltv_months:.1f} months * ${config.twitch_net_sub_take:.2f}) = "
            f"+${item['sub_uplift_margin']:.2f}"
        )
        print(f"- Gifter Reaction Value: +${c.gifter_reaction_value:.2f}")
        
        divider_length = 71
        print("-" * divider_length)
        
        is_winner = c.name == winner_name
        tag = " (WINNER)" if is_winner else ""
        print(f"NET EXPECTED VALUE: +${item['net_ev']:.2f}{tag}")
        print()

    # Print summary leaderboard
    print("=" * 71)
    print("ALGORITHM RECIPIENT PRIORITY RANKING")
    print("=" * 71)
    for rank, item in enumerate(ranked, start=1):
        c = item["candidate"]
        print(f"{rank}. {c.name.split(':')[0]}: Net EV = +${item['net_ev']:.2f} ({c.status})")
    print("=" * 71)


if __name__ == "__main__":
    channel_settings = ChannelConfig()
    run_simulation(CANDIDATES, channel_settings)

Candidate A: The Cold/Disengaged Account
- Status: Offline follower, not in chat.
- Watch Time: 0.5 hrs/mo.
- P(organic) = 0.05% | P(resub|gift) = 0.30% | Uplift = +0.25%
- Ad-block: No (α = 0) | Lost Ad Margin: 0.5 * 6 * $0.0054 = -$0.02
- Sub Uplift: 0.0025 * (3.0 months * $2.50) = +$0.02
- Gifter Reaction Value: +$0.00
-----------------------------------------------------------------------
NET EXPECTED VALUE: +$0.00

Candidate B: The High-Engagement Ad-Tolerant Chatter (No Ad-Block)
- Status: In chat, active chatter, watches 35 hrs/mo on mobile (no ad-block).
- Watch Time: 35.0 hrs/mo.
- P(organic) = 12.00% | P(resub|gift) = 18.00% | Uplift = +6.00%
- Ad-block: No (α = 0) | Lost Ad Margin: 35.0 * 6 * $0.0054 = -$1.13
- Sub Uplift: 0.0600 * (3.2 months * $2.50) = +$0.48
- Gifter Reaction Value: +$0.75
-----------------------------------------------------------------------
NET EXPECTED VALUE: +$0.10

Candidate C: The Persuadable Lurker/Chatter with Ad-Block
- Status: Live in stream ri